<a href="https://colab.research.google.com/github/duttaprat/BMI_511/blob/main/2026/GLM/Intro_to_Genomic_Language_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1: Introduction to Genomic Language Models

**BMI 511 — Spring 2026**
**Instructor:** Pratik Dutta, Ph.D. | Department of Biomedical Informatics, Stony Brook University

---

## Learning objectives

By the end of this notebook you will be able to:

1. Explain what a **Genomic Language Model (gLM)** is and how it differs from classical sequence features.
2. Compare the three main **tokenization** strategies used in gLMs: character, k-mer, and Byte-Pair Encoding (BPE).
3. Load a **pretrained DNABERT-2** model from HuggingFace and run it on a DNA sequence.
4. Inspect token IDs, attention masks, and hidden states.

> Runtime: Runtime → Change runtime type → **T4 GPU** (recommended, not required for this notebook).


## 1. Why do we need language models for DNA?

A genome is a ~3-billion character string over the alphabet `{A, C, G, T, N}`. For decades, we engineered features by hand: GC content, k-mer frequency, PWM scores, conservation tracks. These work, but they are **shallow** — they do not capture long-range context.

Language models turn this around. Instead of hand-crafting features, we train a large neural network to predict masked nucleotides (like BERT on English text). The model is **forced** to learn syntax and semantics of DNA: codon usage, splice signals, TF motifs, chromatin context.

A useful analogy:

| English | DNA |
| --- | --- |
| character | nucleotide (A/C/G/T) |
| word | k-mer or BPE token |
| sentence | gene, promoter, enhancer |
| grammar | biological constraints |
| context | neighboring sequence |

The resulting pretrained model can then be **fine-tuned** on downstream tasks with very little labeled data — the same recipe that powers ChatGPT, just with DNA.

## 2. A short tour of gLMs

| Model | Max length | Tokenization | Training data | Best for |
| --- | --- | --- | --- | --- |
| **DNABERT** (2021) | 512 bp | 3/4/5/6-mer | Human genome | First gLM, easy to use |
| **DNABERT-2** (2024) | 512 tokens (~2–10 kb) | BPE | Multi-species | General purpose, strong baseline |
| **Nucleotide Transformer** (2023) | 1,000 tokens | 6-mer | 850+ species | Long sequences, multi-species |
| **HyenaDNA** (2023) | 1 Mb | Character | Human genome | Ultra-long context |
| **Enformer** (2021) | 200 kb | One-hot (not LM) | Multi-species | Gene expression prediction |
| **Caduceus / Evo** (2024) | 131 kb+ | Character | Prokaryotes/eukaryotes | Generative + predictive |

We will use **DNABERT-2** throughout this module because it is compact, fast, and well-benchmarked.

## 3. Setup

In [1]:
# Install the libraries we need. Run this cell once per Colab session.
!pip install -q transformers==4.44.2 datasets einops biopython scikit-learn umap-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 47.8 MB/s eta 0:00:00


In [2]:

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel

sns.set_style("whitegrid")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {device}")
if device == "cuda":
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

PyTorch  : 2.10.0+cu128
Device   : cuda
GPU      : Tesla T4


## 4. Tokenization — turning DNA into numbers

A neural network cannot read the letter `A`. We first map the sequence to a list of integer **token IDs**. Three strategies dominate the field:

### 4.1 Character-level
Every nucleotide is its own token.
```
ATCGATCG  →  A | T | C | G | A | T | C | G  →  [1, 4, 2, 3, 1, 4, 2, 3]
```
Pros: simple, no information loss, works for long context models (HyenaDNA).
Cons: very long sequences, no motif-level abstraction.

### 4.2 K-mer (overlapping)
We slide a window of size `k` across the sequence.
```
k=3, ATCGATCG  →  ATC | TCG | CGA | GAT | ATC | TCG
```
Pros: captures short motifs (e.g. `ATG` start codon). Used in the original DNABERT.
Cons: vocabulary grows as 4^k; overlapping windows inflate sequence length.

### 4.3 Byte-Pair Encoding (BPE)
BPE **learns** the most frequent substrings from a large corpus and uses them as tokens. This is what DNABERT-2 uses, and what GPT/LLaMA use for text.
```
ATCGATCGATCGATCG  →  ATCGATCG | ATCGATCG    (if ATCGATCG was a frequent merge)
```
Pros: variable-length, compression-efficient, handles any input.
Cons: tokens no longer correspond to fixed biological units.

In [3]:
# Illustrate all three tokenization schemes on the same sequence.
sample_seq = "ATTATATGCATGCTATTGCGGATTGCTGTGCTATTACTCTTCTTGGATTACTTTAGCTGCATGCTAGCTTCAGTGGG"
print(f"Sample ({len(sample_seq)} bp):\n{sample_seq}\n")

# ---- character-level ----
char_tokens = list(sample_seq)
print(f"[char ] n_tokens = {len(char_tokens):3d}   first 10: {char_tokens[:10]}")

# ---- k-mer (k=6, overlapping, like original DNABERT) ----
def kmer_tokens(seq, k=6):
    return [seq[i:i+k] for i in range(len(seq) - k + 1)]

kmers6 = kmer_tokens(sample_seq, k=6)
print(f"[6mer ] n_tokens = {len(kmers6):3d}   first 5 : {kmers6[:5]}")

Sample (77 bp):
ATTATATGCATGCTATTGCGGATTGCTGTGCTATTACTCTTCTTGGATTACTTTAGCTGCATGCTAGCTTCAGTGGG

[char ] n_tokens =  77   first 10: ['A', 'T', 'T', 'A', 'T', 'A', 'T', 'G', 'C', 'A']
er ] n_tokens =  72   first 5 : ['ATTATA', 'TTATAT', 'TATATG', 'ATATGC', 'TATGCA']


In [4]:
# ---- BPE (DNABERT-2) ----
# DNABERT-2 ships a learned BPE tokenizer with ~4,096 merges.
tok = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)

ids = tok(sample_seq, return_tensors="pt")["input_ids"][0]
bpe_tokens = tok.convert_ids_to_tokens(ids)
print(f"[BPE  ] n_tokens = {len(bpe_tokens):3d}")
print(f"         first 10 tokens     : {bpe_tokens[:10]}")
print(f"         token lengths (bp)  : {[len(t) for t in bpe_tokens[:10]]}")
print(f"         vocabulary size     : {tok.vocab_size}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[BPE  ] n_tokens =  19
         first 10 tokens     : ['[CLS]', 'A', 'TTATATG', 'CATG', 'CTATT', 'GC', 'GGATT', 'GCTGTG', 'CTATTA', 'CTCTT']
         token lengths (bp)  : [5, 1, 7, 4, 5, 2, 5, 6, 6, 5]
         vocabulary size     : 4096


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


**What to notice:**

* Character tokenization gives one token per base → sequence length = 76.
* 6-mer gives 71 overlapping tokens → redundant but motif-aware.
* BPE gives far fewer tokens, each covering **a variable number of base pairs**. The tokenizer has learned that some substrings co-occur frequently and deserve their own ID.

The special tokens `[CLS]` (start) and `[SEP]` (end) are added automatically; the first hidden state of `[CLS]` is commonly used as a whole-sequence embedding.

## 5. Load the pretrained DNABERT-2 model

DNABERT-2 is a 117M-parameter BERT-style encoder. We load it straight from the HuggingFace Hub. The first call will download ~450 MB of weights.

> **Compatibility note.** DNABERT-2 ships a custom `BertConfig` via its remote code. Recent `transformers` versions raise a `ValueError` when `AutoModel.from_pretrained(..., trust_remote_code=True)` detects a mismatch between the built-in and the remote config class. The workaround is to load the weights as a standard `BertModel` with an explicit `BertConfig` — this skips the remote config path entirely and gives us the exact same encoder.

In [5]:
# Load DNABERT-2 as a standard BertModel to avoid the config-class mismatch
# that AutoModel raises on newer transformers versions.
from transformers import BertModel, BertConfig

MODEL_NAME = "zhihan1996/DNABERT-2-117M"

# Pull the config first, then load the weights under the stock BertModel class.
config = BertConfig.from_pretrained(MODEL_NAME)
model = BertModel.from_pretrained(
    MODEL_NAME,
    config=config,
    trust_remote_code=True,
).to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"DNABERT-2 loaded. Parameters: {n_params/1e6:.1f} M")
print(f"Hidden size : {model.config.hidden_size}")
print(f"Num layers  : {model.config.num_hidden_layers}")
print(f"Num heads   : {model.config.num_attention_heads}")

config.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


pytorch_model.bin:   0%|          | 0.00/468M [00:00<?, ?B/s]

Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.embeddings.position_embeddings.weight', 'bert.encoder.layer.0.attention.self.key.bias', 'bert.encoder.layer.0.attention.self.key.weight', 'bert.encoder.layer.0.attention.self.query.bias', 'bert.encoder.layer.0.attention.self.query.weight', 'bert.encoder.layer.0.attention.self.value.bias', 'bert.encoder.layer.0.attention.self.value.weight', 'bert.encoder.layer.0.intermediate.dense.bias', 'bert.encoder.layer.0.intermediate.dense.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.dense.bias', 'bert.encoder.layer.0.output.dense.weight', 'bert.encoder.layer.1.attention.self.key.bias', 'bert.encoder.layer.1.attention.self.key.weight', 'bert.encoder.layer.1.attention.self.query.bias', 'bert.encoder.layer.1.attention.self.query.weight', 'bert.encoder.layer.1.attention.self.value.b

DNABERT-2 loaded. Parameters: 89.2 M
Hidden size : 768
Num layers  : 12
Num heads   : 12


## 6. Run the model on a DNA sequence

A forward pass returns three things we care about:

1. `last_hidden_state` — a tensor of shape `(batch, seq_len, hidden_size)`. One vector per token, per sequence.
2. The **mean** (or `[CLS]`) of this tensor → a single vector per sequence → what we use as an embedding.
3. (Optional) attention weights — useful for interpretability.

In [6]:
def embed_sequence(seq, model, tokenizer, device):
    inputs = tokenizer(seq, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    # DNABERT-2 returns a tuple; the first element is last_hidden_state.
    hidden = outputs[0] if isinstance(outputs, tuple) else outputs.last_hidden_state
    # Mean-pool over tokens to get one vector per sequence.
    seq_embedding = hidden.mean(dim=1).squeeze().cpu().numpy()
    return hidden.squeeze().cpu().numpy(), seq_embedding

token_states, seq_vec = embed_sequence(sample_seq, model, tok, device)
print(f"Per-token states : {token_states.shape}   # (n_tokens, 768)")
print(f"Sequence vector  : {seq_vec.shape}         # (768,)")
print(f"First 8 dims     : {seq_vec[:8]}")

Per-token states : (19, 768)   # (n_tokens, 768)
Sequence vector  : (768,)         # (768,)
First 8 dims     : [-1.0305418  -0.43624422  0.37267122 -0.3359768   1.5506777   1.40928
 -0.52719384  1.0279734 ]


## 7. Your turn — short exercise

Use the `embed_sequence` helper to embed the three short sequences below and answer:

1. What are the L2 norms of the three sequence embeddings?
2. Which two sequences are **most similar** by cosine similarity? Does that match your biological intuition?


In [7]:
from sklearn.metrics.pairwise import cosine_similarity

seq_A = "TATAAAAGGCGCGCGCGAATTCGATCGATCGATCGATCG"        # TATA box-like
seq_B = "TATAAATGGCGCGCGCGAATTCGATCGATCGATCGATCG"        # 1 bp mismatch from A
seq_C = "GGGGCCCCGGGGCCCCGGGGCCCCGGGGCCCCGGGGCCCC"        # very different composition

embeds = np.stack([embed_sequence(s, model, tok, device)[1] for s in [seq_A, seq_B, seq_C]])
print("L2 norms :", np.linalg.norm(embeds, axis=1).round(3))
print("Cosine similarity matrix:")
print(np.round(cosine_similarity(embeds), 3))

L2 norms : [27.444 27.474 27.602]
Cosine similarity matrix:
[[1.    0.997 0.977]
 [0.997 1.    0.969]
 [0.977 0.969 1.   ]]


## 8. Recap

* A **genomic language model** is a transformer trained on DNA with a masked-language-model objective.
* **Tokenization** is the first design choice and shapes what the model can learn. DNABERT-2 uses BPE.
* A single forward pass gives us **(n_tokens, 768)** hidden states; mean-pooling yields a 768-d sequence embedding.
* These embeddings can be used directly for similarity search, clustering, or as input to a small classifier.

### What's next

In **Notebook 2** we will extract embeddings for many sequences, visualize them with PCA / UMAP, and see that biologically similar regions cluster together *without any labels*.

In **Notebook 3** we will fine-tune DNABERT-2 to predict whether a sequence is a promoter.

In **Notebook 4** we will use the model to predict the **functional impact of a single-nucleotide variant** — the core idea behind DeepVRegulome.